# NOKA OCR → Supabase
Upload PDF, OCR, langsung masuk ke Supabase.

In [ ]:
# 1. Install dependencies
!apt-get update -qq && apt-get install -y -qq poppler-utils tesseract-ocr
!pip install pytesseract pdf2image opencv-python-headless supabase -q

In [ ]:
# 2. Konfigurasi Supabase
# GANTI URL dan KEY dengan milik lo dari Supabase Dashboard > Project Settings > API
SUPABASE_URL = "https://xxxxx.supabase.co"
SUPABASE_KEY = "service_role_key_lo"

from supabase import create_client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
# 3. Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"File: {pdf_path}")

In [ ]:
# 4. OCR + Indexing langsung ke Supabase
import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path, pdfinfo_from_path
import time, uuid

DOC_ID = str(uuid.uuid4())

supabase.table("ocr_documents").insert({
    "id": DOC_ID,
    "doc_name": pdf_path,
    "status": "processing"
}).execute()

def normalize(text):
    return text.upper().replace("O", "0").replace("I", "1").replace("B", "8")

def preprocess(pil_image):
    img = np.array(pil_image.convert("L"))
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary = cv2.resize(binary, None, fx=1.2, fy=1.2, interpolation=cv2.INTER_LINEAR)
    return binary

def ocr_page(page_num, processed_img):
    data = pytesseract.image_to_data(processed_img, config="--psm 6", output_type=Output.DICT)
    rows = []
    for i, text in enumerate(data["text"]):
        if text.strip():
            rows.append({
                "doc_id": DOC_ID,
                "page_num": page_num,
                "word": text.strip(),
                "word_normalized": normalize(text.strip()),
                "confidence": int(data["conf"][i]),
                "x": int(data["left"][i]),
                "y": int(data["top"][i]),
                "w": int(data["width"][i]),
                "h": int(data["height"][i]),
            })
    return rows

info = pdfinfo_from_path(pdf_path)
total_pages = info["Pages"]
print(f"Total halaman: {total_pages}")

total_words = 0; start_time = time.time()
for start in range(1, total_pages + 1, 10):
    end = min(start + 9, total_pages)
    pages = convert_from_path(pdf_path, dpi=100, first_page=start, last_page=end)
    for i, page in enumerate(pages):
        page_num = start + i
        words = ocr_page(page_num, preprocess(page))
        if words:
            supabase.table("ocr_words").insert(words).execute()
        total_words += len(words)
        print(f"  Page {page_num}/{total_pages} -> {len(words)} kata")

elapsed = time.time() - start_time
supabase.table("ocr_documents").update({
    "status": "completed",
    "total_pages": total_pages,
    "total_words": total_words
}).eq("id", DOC_ID).execute()

print(f"\nSelesai! {total_words} kata dari {total_pages} halaman dalam {elapsed:.1f}s")
print(f"Doc ID: {DOC_ID}")

## Selesai!
Sekarang buka frontend (Vercel) atau backend lokal buat search.